# Parallel Coordinates Plots

### Visualization of scenario metrics using parallel coordinates plots.

##### OUTDATED: The data used in this notebook is not in the repo. contact canruso@berkeley.edu | dinobellugi@berkeley.edu if you want to see the data and reproduce the plots.

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

import sys
sys.path.append('./coeqwalpackage')

from coeqwalpackage.plotting import (
    reorganize_objs,
    get_color,
    get_zorder,
    custom_parallel_coordinates_highlight_scenarios,
    custom_parallel_coordinates_highlight_scenarios_baseline_at_zero,
    custom_parallel_coordinates_highlight_variability,
    custom_parallel_coordinates_highlight_quantile,
    custom_parallel_coordinates_highlight_iqr,
    custom_parallel_coordinates_highlight_cluster,
)
from coeqwalpackage.metrics import (
    percent_change_from_baseline_by_index,
    process_scenario_dataframe,
    calculate_scenario_statistics,
)

## Configuration

In [2]:
# =============================================================================
# NOTEBOOK CONFIGURATION
# =============================================================================

# Paths
DATA_DIR_METRICS = '../output/metrics/'
DATA_DIR_KNOBS = '../data/parallelplots/'
OUTPUT_DIR = '../output/parallelplots/'

# Create output directory if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Helper to get full output path
def out(filename):
    """Return full path for output file."""
    return os.path.join(OUTPUT_DIR, filename)

# Figure settings
FIGSIZE = (18, 6)
FIGSIZE_LARGE = (22, 8)
FONTSIZE = 14
DPI = 300

# Colors for clusters
MEDIAN_COLORS = ['#DC143C', '#FF8C00', '#4169E1']  # Crimson, Orange, RoyalBlue
CLUSTER_COLORS = {1: 'firebrick', 2: 'goldenrod', 3: 'cornflowerblue'}

# Scenario configuration
HIGHLIGHT_INDICES = [0, 4, 15, 261, 320, 360]
HIGHLIGHT_DESCRIPTIONS = [
    'Business As Usual',
    'Higher Flows',
    'A Saltier Delta',
    'New Balance',
    'Carry It Forward',
    'Prioritized Drinking Water'
]
HIGHLIGHT_COLORS = ['black', 'red', 'blue', 'green', 'orange', 'purple']

# Column definitions for IQR metrics
IQR_DEL_COLS = [
    'iqr_Ann_Avg_SWPTotDel_TAF', 'iqr_Ann_Avg_SWPMIDel_TAF', 'iqr_Ann_Avg_CVPTotDel_TAF',
    'iqr_Ann_Avg_CVPAgDel_TAF', 'iqr_Ann_Avg_CVPSCEXDel_TAF', 'iqr_Ann_Avg_BanksJones_TAF'
]
IQR_REST_COLS = [
    'iqr_Apr_Avg_ShstaStorage_TAF', 'iqr_Apr_Avg_OrovlStorage_TAF',
    'iqr_Fall_Avg_X2_KM', 'iqr_Spring_Avg_X2_KM', 'iqr_Ann_Avg_NDO_TAF'
]

# Wet IQR columns (with 'Wet_' prefix)
IQR_WET_DEL_COLS = [c.replace('iqr_', 'iqr_Wet_') for c in IQR_DEL_COLS]
IQR_WET_REST_COLS = [c.replace('iqr_', 'iqr_Wet_') for c in IQR_REST_COLS]

# Dry IQR columns (with 'Dry_' prefix)
IQR_DRY_DEL_COLS = [c.replace('iqr_', 'iqr_Dry_') for c in IQR_DEL_COLS]
IQR_DRY_REST_COLS = [c.replace('iqr_', 'iqr_Dry_') for c in IQR_REST_COLS]

# New metrics columns
NEW_METRICS_COLS = [
    'DEL_NOD_AG_TOTAL_TAF', 'DEL_SJV_AG_TOTAL_TAF', 'DEL_NOD_MI_TOTAL_TAF',
    'DEL_SJV_MI_TOTAL_TAF', 'DEL_SOCAL_MI_TOTAL_TAF', 'CVP_SWP_EXPORTS_TAF',
    'NDO_TAF', 'SAC_IN_TAF', 'SJR_IN_TAF', 'X2_APR_KM', 'X2_OCT_KM',
    'STO_NOD_TOTAL_OCT_TAF', 'STO_SOD_TOTAL_OCT_TAF'
]

NEW_METRICS_LABELS = [
    'Sac Valley AG Deliveries', 'SJ Valley AG Deliveries', 'Sac Valley Municipal Deliveries',
    'SJ Valley Municipal Deliveries', 'SoCal Municipal Deliveries', 'Delta Exports',
    'Delta Outflows', 'Sac River Inflows', 'SJ River Inflows', 'X2 Salinity (Apr)',
    'X2 Salinity (Oct)', 'North of Delta Storage (Sep)', 'South of Delta Storage (Sep)'
]

print(f"Output directory: {OUTPUT_DIR}")

Output directory: ../output/parallelplots/


## Load Cluster Data

In [3]:
# Load cluster assignments
clusters = {
    'all': pd.read_csv(DATA_DIR_KNOBS + 'filtered_metrics_07_01_2024_catknobs_KmedIdx3.csv')['Cluster'],
    'wet': pd.read_csv(DATA_DIR_KNOBS + 'filtered_wet_metrics_07_01_2024_catknobs_KmedIdx3.csv')['Cluster'],
    'dry': pd.read_csv(DATA_DIR_KNOBS + 'filtered_dry_metrics_07_01_2024_catknobs_KmedIdx3.csv')['Cluster'],
}

# Load medoid knobs
knobs = {
    'all': pd.read_csv(DATA_DIR_KNOBS + 'AllMedoidsStudyKnobs.csv'),
    'wet': pd.read_csv(DATA_DIR_KNOBS + 'WetMedoidsStudyKnobs.csv'),
    'dry': pd.read_csv(DATA_DIR_KNOBS + 'DryMedoidsStudyKnobs.csv'),
}

medoids = {k: v['Study'].tolist() for k, v in knobs.items()}
print(f'Loaded clusters and medoids for: {list(clusters.keys())}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/parallelplots/filtered_metrics_07_01_2024_catknobs_KmedIdx3.csv'

## Load IQR Metrics

In [ ]:
# Load IQR data
iqr_data = {}
for variant in ['all', 'wet', 'dry']:
    suffix = '' if variant == 'all' else f'_{variant}'
    filename = f'filtered_iqr{suffix}_metrics_08_11_2024.csv'
    df = pd.read_csv(DATA_DIR_METRICS + filename).drop(['Unnamed: 0'], axis=1, errors='ignore').iloc[:, 6:]
    df['Cluster'] = clusters[variant]
    iqr_data[variant] = df

print(f'IQR data shapes: {[(k, v.shape) for k, v in iqr_data.items()]}')

In [ ]:
# Prepare IQR subsets by column groups and clusters
def get_iqr_subsets(data_dict, del_cols, rest_cols, variant_prefix=''):
    """Extract delivery and rest columns, split by cluster."""
    result = {}
    for variant, df in data_dict.items():
        prefix = variant_prefix if variant == 'all' else f'{variant.capitalize()}_'
        
        # Get the right column set based on variant
        if variant == 'all':
            dcols, rcols = del_cols, rest_cols
        elif variant == 'wet':
            dcols = [c.replace('iqr_', 'iqr_Wet_') for c in del_cols]
            rcols = [c.replace('iqr_', 'iqr_Wet_') for c in rest_cols]
        else:  # dry
            dcols = [c.replace('iqr_', 'iqr_Dry_') for c in del_cols]
            rcols = [c.replace('iqr_', 'iqr_Dry_') for c in rest_cols]
        
        dcols_present = [c for c in dcols if c in df.columns]
        rcols_present = [c for c in rcols if c in df.columns]
        
        result[f'{variant}_del'] = df[dcols_present + ['Cluster']]
        result[f'{variant}_rest'] = df[rcols_present + ['Cluster']]
        
        # Split by cluster
        for cluster in [1, 2, 3]:
            result[f'{variant}_del_cl{cluster}'] = df[df['Cluster'] == cluster][dcols_present]
            result[f'{variant}_rest_cl{cluster}'] = df[df['Cluster'] == cluster][rcols_present]
    
    return result

iqr_subsets = get_iqr_subsets(iqr_data, IQR_DEL_COLS, IQR_REST_COLS)
print(f'Created {len(iqr_subsets)} IQR subsets')

## IQR Cluster Plots

In [ ]:
# Plot IQR metrics by cluster
# This replaces ~25 individual plotting cells

plot_configs = []
for variant in ['all', 'wet', 'dry']:
    for group in ['del', 'rest']:
        key = f'{variant}_{group}'
        df = iqr_subsets[key]
        ncols = len(df.columns) - 1  # minus Cluster
        plot_configs.append({
            'data': df,
            'title': f'IQR {group.upper()} - {variant.upper()}',
            'minmaxs': ['max'] * ncols
        })
        
        # Per-cluster plots
        for cluster in [1, 2, 3]:
            cl_key = f'{variant}_{group}_cl{cluster}'
            cl_df = iqr_subsets[cl_key]
            plot_configs.append({
                'data': cl_df,
                'title': f'IQR {group.upper()} - {variant.upper()} - Cluster {cluster}',
                'minmaxs': ['max'] * len(cl_df.columns),
                'color': CLUSTER_COLORS[cluster]
            })

print(f'Prepared {len(plot_configs)} plot configurations')

In [ ]:
# Execute cluster plots (uncomment to run)
# for cfg in plot_configs:
#     df = cfg['data']
#     cols = [c for c in df.columns if c != 'Cluster']
#     custom_parallel_coordinates_highlight_cluster(
#         df[cols] if 'Cluster' not in df.columns else df,
#         columns_axes=cols,
#         axis_labels=cols,
#         zorder_by=4,
#         ideal_direction='top',
#         alpha_base=0.8,
#         lw_base=1.5,
#         fontsize=FONTSIZE,
#         figsize=FIGSIZE,
#         minmaxs=cfg['minmaxs'],
#         color_dict_categorical=CLUSTER_COLORS,
#     )
#     print(f"Plotted: {cfg['title']}")

## Load New Scenario Metrics

In [ ]:
# Load scenario metrics (all/wet/dry)
def load_new_metrics(filename_pattern, cols, labels, suffix=''):
    """Load and rename metrics dataframe."""
    df = pd.read_csv(DATA_DIR_KNOBS + filename_pattern).drop(['Unnamed: 0'], axis=1, errors='ignore').iloc[:, 5:]
    # Use flexible column selection
    cols_wet = [c.replace('_TAF', '_TAF').replace('_KM', '_KM') for c in cols]
    cols_present = [c for c in cols_wet if c in df.columns]
    if len(cols_present) == len(labels):
        df = df[cols_present]
        df.columns = [f'{l}{suffix}' for l in labels]
    return df

# Adjust column names for wet/dry (Sep vs Oct storage)
NEW_METRICS_COLS_WET = NEW_METRICS_COLS.copy()
NEW_METRICS_COLS_WET[-2] = 'STO_NOD_TOTAL_SEP_TAF'
NEW_METRICS_COLS_WET[-1] = 'STO_SOD_TOTAL_SEP_TAF'

new_metrics = {
    'all': load_new_metrics('20240829DRAFT_ALL_AggFlowDel_ANNUAL_AVGS.csv', NEW_METRICS_COLS, NEW_METRICS_LABELS),
    'wet': load_new_metrics('20240829DRAFT_ALL_AggFlowDel_ANNUAL_AVGS_WET.csv', NEW_METRICS_COLS_WET, NEW_METRICS_LABELS, ' (Wet)'),
    'dry': load_new_metrics('20240829DRAFT_ALL_AggFlowDel_ANNUAL_AVGS_DRY.csv', NEW_METRICS_COLS_WET, NEW_METRICS_LABELS, ' (Dry)'),
}

print(f'Loaded new metrics: {[(k, v.shape) for k, v in new_metrics.items()]}')

In [ ]:
# Load time series for statistics
ts_files = {
    0: '20240829DRAFT_expl0000_AggFlowDelTS_ANNUAL_VOLUME.csv',
    4: '20240829DRAFT_expl0004_AggFlowDelTS_ANNUAL_VOLUME.csv',
    15: '20240829DRAFT_expl0015_AggFlowDelTS_ANNUAL_VOLUME.csv',
    261: '20240829DRAFT_expl0261_AggFlowDelTS_ANNUAL_VOLUME.csv',
    320: '20240829DRAFT_expl0320_AggFlowDelTS_ANNUAL_VOLUME.csv',
    360: '20240829DRAFT_expl0360_AggFlowDelTS_ANNUAL_VOLUME.csv',
}

ts_data = {}
for idx, fname in ts_files.items():
    ts_data[idx] = pd.read_csv(
        DATA_DIR_KNOBS + fname,
        header=[0, 1, 2, 3, 4, 5, 6],
        index_col=0,
        parse_dates=True
    )

print(f'Loaded {len(ts_data)} time series files')

## Scenario Parallel Plots

In [ ]:
# Plot scenarios - raw values
# This replaces ~12 individual plotting cells

for variant, df in new_metrics.items():
    suffix = '' if variant == 'all' else f' - {variant.capitalize()} Data'
    
    # All scenarios
    custom_parallel_coordinates_highlight_scenarios(
        objs=df,
        columns_axes=df.columns,
        axis_labels=df.columns,
        ideal_direction='top',
        minmaxs=['max'] * len(df.columns),
        color_dict_categorical={1: 'grey'},
        highlight_indices=HIGHLIGHT_INDICES,
        highlight_colors=HIGHLIGHT_COLORS,
        highlight_descriptions=HIGHLIGHT_DESCRIPTIONS,
        save_fig_filename=out(f'parallel_coordinates_{variant}.png'),
        title=f'Parallel Line Plot of All Scenarios{suffix}',
        fontsize=12,
        figsize=FIGSIZE_LARGE,
    )
    
    # Selected scenarios only (using iloc for positional indexing)
    custom_parallel_coordinates_highlight_scenarios(
        objs=df.iloc[HIGHLIGHT_INDICES],
        columns_axes=df.columns,
        axis_labels=df.columns,
        ideal_direction='top',
        minmaxs=['max'] * len(df.columns),
        color_dict_categorical={1: 'grey'},
        highlight_indices=HIGHLIGHT_INDICES,
        highlight_colors=HIGHLIGHT_COLORS,
        highlight_descriptions=HIGHLIGHT_DESCRIPTIONS,
        save_fig_filename=out(f'parallel_coordinates_{variant}_selected.png'),
        title=f'Parallel Line Plot of Selected Scenarios{suffix}',
        fontsize=12,
        figsize=FIGSIZE_LARGE,
    )
    print(f'Saved: {variant} plots to {OUTPUT_DIR}')

## Percent Change Plots

In [ ]:
# Compute and plot percent change from baseline
# This replaces ~6 cells of repeated percent change calculations and plots

for variant, df in new_metrics.items():
    suffix = '' if variant == 'all' else f' - {variant.capitalize()} Data'
    
    # Calculate percent change
    pct_df = percent_change_from_baseline_by_index(df, baseline_index=0)
    
    # All scenarios
    custom_parallel_coordinates_highlight_scenarios_baseline_at_zero(
        objs=pct_df,
        columns_axes=pct_df.columns,
        axis_labels=pct_df.columns,
        highlight_indices=HIGHLIGHT_INDICES,
        highlight_colors=HIGHLIGHT_COLORS,
        highlight_descriptions=HIGHLIGHT_DESCRIPTIONS,
        save_fig_filename=out(f'parallel_coordinates_percent_changes_{variant}.png'),
        title=f'Parallel Line Plot of Percent Change From Baseline{suffix}',
        fontsize=12,
        figsize=FIGSIZE_LARGE,
    )
    
    # Selected scenarios (using iloc for consistency)
    custom_parallel_coordinates_highlight_scenarios_baseline_at_zero(
        objs=pct_df.iloc[HIGHLIGHT_INDICES],
        columns_axes=pct_df.columns,
        axis_labels=pct_df.columns,
        highlight_indices=HIGHLIGHT_INDICES,
        highlight_colors=HIGHLIGHT_COLORS,
        highlight_descriptions=HIGHLIGHT_DESCRIPTIONS,
        save_fig_filename=out(f'parallel_coordinates_percent_changes_{variant}_selected.png'),
        title=f'Parallel Line Plot of Percent Change From Baseline - Selected{suffix}',
        fontsize=12,
        figsize=FIGSIZE_LARGE,
    )
    print(f'Saved: {variant} percent change plots')

## Variability and Quantile Plots

In [ ]:
# Calculate statistics for highlighted scenarios
dataframes = [ts_data[idx] for idx in HIGHLIGHT_INDICES]
median_data, std_data, p90_data, p10_data = calculate_scenario_statistics(
    dataframes, HIGHLIGHT_DESCRIPTIONS
)

# Set index to match highlight indices for plotting
for df in [median_data, std_data, p90_data, p10_data]:
    df.index = HIGHLIGHT_INDICES

print(f'Statistics computed: median={median_data.shape}, std={std_data.shape}')

In [ ]:
# Pairwise variability plots (Baseline vs each scenario)
# This replaces ~6 individual cells

baseline_idx = HIGHLIGHT_INDICES[0]
for i, (idx, color, desc) in enumerate(zip(HIGHLIGHT_INDICES[1:], HIGHLIGHT_COLORS[1:], HIGHLIGHT_DESCRIPTIONS[1:])):
    pair_indices = [baseline_idx, idx]
    pair_colors = ['black', color]
    pair_descs = [HIGHLIGHT_DESCRIPTIONS[0], desc]
    
    custom_parallel_coordinates_highlight_variability(
        objs=new_metrics['all'].loc[pair_indices],
        variability_data=std_data,
        highlight_indices=pair_indices,
        highlight_colors=pair_colors,
        highlight_descriptions=pair_descs,
        save_fig_filename=out(f'parallel_coordinates_std_{i+1}.png'),
        title=f'{HIGHLIGHT_DESCRIPTIONS[0]} vs {desc}: Means ± 1 Std Dev',
        figsize=FIGSIZE_LARGE,
        fontsize=12,
    )
    print(f'Saved: variability plot {i+1}')

# All scenarios variability
custom_parallel_coordinates_highlight_variability(
    objs=new_metrics['all'].loc[HIGHLIGHT_INDICES],
    variability_data=std_data,
    highlight_indices=HIGHLIGHT_INDICES,
    highlight_colors=HIGHLIGHT_COLORS,
    highlight_descriptions=HIGHLIGHT_DESCRIPTIONS,
    save_fig_filename=out('parallel_coordinates_std_all.png'),
    title='Selected Scenarios: Means ± 1 Standard Deviation',
    figsize=FIGSIZE_LARGE,
    fontsize=12,
)
print('Saved: all variability plot')

In [ ]:
# Pairwise quantile plots (Baseline vs each scenario)
# This replaces ~7 individual cells

for i, (idx, color, desc) in enumerate(zip(HIGHLIGHT_INDICES[1:], HIGHLIGHT_COLORS[1:], HIGHLIGHT_DESCRIPTIONS[1:])):
    pair_indices = [baseline_idx, idx]
    pair_colors = ['black', color]
    pair_descs = [HIGHLIGHT_DESCRIPTIONS[0], desc]
    
    custom_parallel_coordinates_highlight_quantile(
        objs=median_data.loc[pair_indices],
        lower_bound_data=p10_data,
        upper_bound_data=p90_data,
        highlight_indices=pair_indices,
        highlight_colors=pair_colors,
        highlight_descriptions=pair_descs,
        save_fig_filename=out(f'parallel_coordinates_quantiles_{i+1}.png'),
        title=f'{HIGHLIGHT_DESCRIPTIONS[0]} vs {desc}: Medians with 10th/90th Percentiles',
        figsize=FIGSIZE_LARGE,
        fontsize=12,
    )
    print(f'Saved: quantile plot {i+1}')

# All scenarios quantiles
custom_parallel_coordinates_highlight_quantile(
    objs=median_data.loc[HIGHLIGHT_INDICES],
    lower_bound_data=p10_data,
    upper_bound_data=p90_data,
    highlight_indices=HIGHLIGHT_INDICES,
    highlight_colors=HIGHLIGHT_COLORS,
    highlight_descriptions=HIGHLIGHT_DESCRIPTIONS,
    save_fig_filename=out('parallel_coordinates_quantiles_all.png'),
    title='Selected Scenarios: Medians with 10th and 90th Percentiles',
    figsize=FIGSIZE_LARGE,
    fontsize=12,
)
print('Saved: all quantiles plot')

In [ ]:
print('Done!')